In [1]:
from sqlalchemy import MetaData
from sqlalchemy import Table, Column, Integer, String, Float, text
from sqlalchemy import create_engine

import pandas as pd

from uuid import uuid4

from dotenv import load_dotenv

from sentence_transformers import CrossEncoder

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from qdrant_client import QdrantClient
from qdrant_client.http.models import VectorParams, Distance

from langchain_qdrant import QdrantVectorStore
from langchain_core.documents import Document
from langchain.tools import tool

import tqdm as notebook_tqdm

from langfuse import get_client
from langfuse.langchain import CallbackHandler
import os


load_dotenv()

/home/hasyim/projects-ai-engineer/Capston3/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
langfuse = get_client()
langfuse_handler = CallbackHandler()

In [3]:
data_path = "/home/hasyim/Bootcamp_AI/capston/Capston3/chatbot/data/raw/imdb_top_1000.csv"
df = pd.read_csv(data_path)

In [4]:
df=df.replace({'Released_Year': 'PG'}, None)

In [5]:
df['Gross'] = df['Gross'].str.replace(',', '', regex=True)

In [6]:
df[['Released_Year','Gross']] = df[['Released_Year','Gross']].apply(pd.to_numeric)

In [7]:
df['film_id'] = [str(uuid4()) for _ in range(len(df['Series_Title']))]

In [7]:
df_clean = df.drop(columns="Overview")

In [42]:
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS FILM_TABEL"))

In [43]:
metadata_obj = MetaData()

In [41]:
engine = create_engine('sqlite:////home/hasyim/Bootcamp_AI/capston/Capston3/chatbot/data/process/capston3.db')

In [44]:
FILM = Table(
    "FILM_TABEL",
    metadata_obj,
    Column("film_id", String, primary_key=True, default=lambda: str(uuid4())),
    Column("Series_Title", String, nullable=False),
    Column("Released_Year", Float),
    Column("Certificate", String),
    Column("Runtime", String),
    Column("Genre", String),
    Column("IMDB_Rating", Float),
    Column("Meta_score", Float),
    Column("Director", String),
    Column("Star1", String),
    Column("Star2", String),
    Column("Star3", String),
    Column("Star4", String),
    Column("No_of_Votes", Integer),
    Column("Gross", Float),
    Column("Poster_Link", String),
    extend_existing=True
)

In [45]:
df_clean.to_sql(
    name='FILM_TABEL',
    con=engine,
    if_exists='append',
    index=False
)

1000

In [46]:
df_new = pd.read_sql('SELECT * FROM FILM_TABEL', con=engine)

In [9]:
df.head(1)

,Poster_Link,Series_Title,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Overview,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross,film_id
0,https://m.media-amazon.com/images/M/MV5BMDFkYT...,The Shawshank Redemption,1994.0,A,142 min,Drama,9.3,Two imprisoned men bond over a number of years...,80.0,Frank Darabont,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler,2343110,28341469.0,ce74ed01-ea88-4179-a0d1-0fc97c17556a


In [5]:
import subprocess

def check_gpu():
    try:
        # Menjalankan perintah nvidia-smi untuk cek koneksi GPU
        subprocess.check_output('nvidia-smi')
        return "cuda"
    except (subprocess.CalledProcessError, FileNotFoundError):
        return "cpu"

device = check_gpu()
print(f"Menggunakan: {device}")

Menggunakan: cuda


In [6]:
embedding = OpenAIEmbeddings(
    model='text-embedding-3-small',
)

rerank = CrossEncoder("Qwen/Qwen3-Reranker-0.6B", device=device, cache_folder="chatbot/model")

url = os.getenv("QDRANT_URL")

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 1354.66it/s]


In [11]:
documents = []

for i in range(len(df)):
    judul_film = df['Series_Title'][i]
    overview_film = df['Overview'][i]
    id_film = df['film_id'][i]
    input_rag = f"Series_Title: {judul_film}, Overview: {overview_film}"
    doc = Document(
        page_content=input_rag,
        metadata={
            "film_id": id_film,
            "Series_Title": judul_film
        },
    )
    documents.append(doc)

In [ ]:
uuids = [str(uuid4()) for _ in range(len(documents))]

In [67]:
client = QdrantClient(
    url=url,
    api_key=os.getenv("QDRANT_API")
)


In [ ]:
qdrant = QdrantVectorStore.from_documents(
    documents,
    embedding=embedding,
    url=url,
    api_key=os.getenv("QDRANT_API"),
    collection_name="coba_capstron3_coba",
)

TypeError: Client.__init__() got an unexpected keyword argument 'client'

In [15]:
client = QdrantClient(
    url=url,
    api_key=os.getenv("QDRANT_API")
)

response = client.get_collections()
print(response)

collections=[CollectionDescription(name='amazon_product'), CollectionDescription(name='coba_capstron3_coba'), CollectionDescription(name='flipkart_products'), CollectionDescription(name='coba_capstron3')]


In [ ]:
retrive = QdrantVectorStore.from_existing_collection(
    embedding=embedding,
    collection_name="coba_capstron3_coba",
    url=url,
    api_key=os.getenv("QDRANT_API"),
)

In [48]:
query = "can you suggest me a film that is related to mafia"
results = retrive.similarity_search(
    query=query,
    k=5,
)
for result in results:
    print(result)
    
hasil_rag = [result.page_content for result in results]

page_content='Series_Title: Donnie Brasco, Overview: An FBI undercover agent infiltrates the mob and finds himself identifying more with the mafia life, at the expense of his regular one.' metadata={'film_id': '4ae686b9-2fa2-43ec-81d1-39fa018cd11c', 'Series_Title': 'Donnie Brasco', '_id': '69998cbd-8343-446a-9968-e844b7d898af', '_collection_name': 'coba_capstron3_coba'}
page_content='Series_Title: The Godfather, Overview: An organized crime dynasty's aging patriarch transfers control of his clandestine empire to his reluctant son.' metadata={'film_id': 'e8193b17-dfc7-44e9-9e34-a3649ff60542', 'Series_Title': 'The Godfather', '_id': 'd32dc821-1214-4260-89e9-db1a1ee10732', '_collection_name': 'coba_capstron3_coba'}
page_content='Series_Title: American Gangster, Overview: An outcast New York City cop is charged with bringing down Harlem drug lord Frank Lucas, whose real life inspired this partly biographical film.' metadata={'film_id': '2402278f-52ad-427f-be87-b60820e478db', 'Series_Title'

In [49]:
reranking = rerank.rank(
    query, 
    hasil_rag,
    return_documents=True, 
    top_k=3
)

In [50]:
reranking

[{'corpus_id': 4,
  'score': np.float32(2.1875),
  'text': 'Series_Title: Goodfellas, Overview: The story of Henry Hill and his life in the mob, covering his relationship with his wife Karen Hill and his mob partners Jimmy Conway and Tommy DeVito in the Italian-American crime syndicate.'},
 {'corpus_id': 1,
  'score': np.float32(1.9375),
  'text': "Series_Title: The Godfather, Overview: An organized crime dynasty's aging patriarch transfers control of his clandestine empire to his reluctant son."},
 {'corpus_id': 0,
  'score': np.float32(1.25),
  'text': 'Series_Title: Donnie Brasco, Overview: An FBI undercover agent infiltrates the mob and finds himself identifying more with the mafia life, at the expense of his regular one.'}]

In [51]:
context_list = [item['text'] for item in reranking]
context_list

['Series_Title: Goodfellas, Overview: The story of Henry Hill and his life in the mob, covering his relationship with his wife Karen Hill and his mob partners Jimmy Conway and Tommy DeVito in the Italian-American crime syndicate.',
 "Series_Title: The Godfather, Overview: An organized crime dynasty's aging patriarch transfers control of his clandestine empire to his reluctant son.",
 'Series_Title: Donnie Brasco, Overview: An FBI undercover agent infiltrates the mob and finds himself identifying more with the mafia life, at the expense of his regular one.']

In [52]:
context_for_llm = "\n\n".join(context_list)
print(context_for_llm)

Series_Title: Goodfellas, Overview: The story of Henry Hill and his life in the mob, covering his relationship with his wife Karen Hill and his mob partners Jimmy Conway and Tommy DeVito in the Italian-American crime syndicate.

Series_Title: The Godfather, Overview: An organized crime dynasty's aging patriarch transfers control of his clandestine empire to his reluctant son.

Series_Title: Donnie Brasco, Overview: An FBI undercover agent infiltrates the mob and finds himself identifying more with the mafia life, at the expense of his regular one.


In [60]:
@tool
def RAG_tool(query: str) -> str:
    """This tools is used to call data from Qdrant based on user query"""
    results = retrive.similarity_search(query=query, k=5)
    hasil_rag = [result.page_content for result in results]
    reranking = rerank.rank(
    query, 
    hasil_rag,
    return_documents=True, 
    top_k=3
    )
    context_list = [item['text'] for item in reranking]
    return context_list
    
tools = [RAG_tool]

In [56]:
print("✅ Tools berhasil dibuat:")
for t in tools:
    print(f"   🔧 {t.name}: {t.description[:60]}...")

✅ Tools berhasil dibuat:
   🔧 RAG_tool: This tools is used to call data from Qdrant based on user qu...


In [61]:
RAG_tool.invoke(query)

['Series_Title: Goodfellas, Overview: The story of Henry Hill and his life in the mob, covering his relationship with his wife Karen Hill and his mob partners Jimmy Conway and Tommy DeVito in the Italian-American crime syndicate.',
 "Series_Title: The Godfather, Overview: An organized crime dynasty's aging patriarch transfers control of his clandestine empire to his reluctant son.",
 'Series_Title: Donnie Brasco, Overview: An FBI undercover agent infiltrates the mob and finds himself identifying more with the mafia life, at the expense of his regular one.']

In [10]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

In [14]:
from langchain.agents import create_agent
from chatbot.tools.tool import tools

SYSTEM_PROMPT = """You are an agent for retrive the document of RAG from user query, your job is to take user query and see what user intention and rewrite to be more efficeint query.
If user query is in langunage other tahn english tranlate it first adn rewrite it before call the tool for RAG
"""

agent_app = create_agent(
    llm,
    tools,
    system_prompt=SYSTEM_PROMPT
)

print("✅ Agent berhasil dibuat!")
print(f"   Model  : {llm.model_name}")
print(f"   Tools  : {[t.name for t in tools]}")

✅ Agent berhasil dibuat!
   Model  : gpt-4o-mini
   Tools  : ['RAG_tool']


In [15]:
from IPython.display import Markdown, display

# Test 1: Pertanyaan Produk
print("=" * 60)
print("TEST 1: Rekomendasi Produk")
print("=" * 60)

response = agent_app.invoke(
    {"messages": [{"role": "user", "content": "Saya penasaran saya kan belajar sejarah disekolah, nah apakah ada perang mengenai perang dunia ke 2?"}]}
)
answer = response["messages"][-1].content
display(Markdown(answer))

TEST 1: Rekomendasi Produk


UnexpectedResponse: Unexpected Response: 404 (Not Found)
Raw response content:
b'{"status":{"error":"Not found: Collection `percobaan_capston3` doesn\'t exist!"},"time":0.000014648}'